# Alcohol Physicochemical Properties — Regression Analysis (SLR / MLR)

This notebook implements the regression methodology, redesigned according to
reviewer feedback on the "New Augmented Vertex-Adjacency Matrices Descriptors..."
manuscript.

**Scope:** 12 physicochemical properties (bp, MW, nD, epsilon, Tc, Vc, Pc, logP, vp,
eta, gamma, lambda), each modeled separately with 6 independent descriptor
groups (CW-I/II/mix x H-Included/H-Suppressed).

**How it works:** First all the Excel data is loaded, then you are asked which
property you want to analyze (1-12). Run each cell in order.

**IMPORTANT:** Set the `EXCEL_PATH` variable to match your own file path (below,
in the Settings cell). The notebook assumes that the `Alcohol_Data.xlsx` file
has the SAME COLUMN HEADERS as your Excel file (All_Properties + 6 CW-* sheets).

## 1. Libraries

In [ ]:
# If a library is missing, uncomment the lines below and run them:
# !pip install pandas numpy matplotlib statsmodels scipy python-docx openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.section import WD_ORIENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

import io
import os

print("Libraries loaded.")

## 2. Settings and Constants

ALL the values here reflect decisions made during months of methodology discussion. I've added a comment next to each one explaining WHY it was chosen that way.

In [ ]:
# --- File path -------------------------------------------------------------
# Enter the path to the Alcohol_Data.xlsx file on your own computer/Colab.
EXCEL_PATH = "Alcohol_Data.xlsx"

# --- VIF threshold ---------------------------------------------------------
# WHY 5: Many sources use VIF>10, but for our small dataset (n drops to as
# low as 13 for some properties) we chose a STRICTER threshold (5) - being
# more cautious with little data increases defensibility.
VIF_THRESHOLD = 5

# --- Candidate descriptors (same 5 candidates for every group) -------------
CANDIDATE_VARIABLES = ['Per(A)', 'A(2,1)', 'A(2,2)', 'tr(A)', 'epsilon(A)']

# --- 6 independent descriptor groups --------------------------------------
# WHY SEPARATE: Each group (CW-I/II/mix x H-Included/H-Suppressed) contains
# descriptors derived from the SAME molecular graph but with a DIFFERENT
# weighting/hydrogen-inclusion scheme - variables are NEVER MIXED across
# groups, each one is evaluated on its own.
# NOTE: The KEYS of this dictionary (group names) are used as the heading
# text in the Word reports - kept in English here since the report/table
# text is in English. Only the group name that appears in the FIGURES (the
# Y-randomization histogram and the Williams Plot) needs to be in English
# for the figures themselves, so there is a SEPARATE ENGLISH_GROUP_NAME
# dictionary below - the plotting functions take the English name ONLY from
# that dictionary. The VALUES (Excel sheet names) MUST match the ACTUAL
# sheet names in the data file exactly - those were not changed.
GROUP_SHEETS = {
    'CW-I (H-Suppressed)': 'CW-I_without_H',
    'CW-II (H-Suppressed)': 'CW-II_without_H',
    'CW-mix (H-Suppressed)': 'CW-mix_without_H',
    'CW-I (H-Included)': 'CW-I_with_H',
    'CW-II (H-Included)': 'CW-II_with_H',
    'CW-mix (H-Included)': 'CW-mix_with_H',
}

# --- English equivalent of the group names, shown ONLY IN THE FIGURES
# (plots). WHY SEPARATE: the text/table/heading in the Word report stays in
# English, but the TEXT ON the figures (plots) to be included in the
# manuscript cannot be edited afterward, so they must be in English. This is
# why the group name is written from this dictionary when a figure is
# plotted.
ENGLISH_GROUP_NAME = {
    'CW-I (H-Suppressed)': 'CW-I (H-Suppressed)',
    'CW-II (H-Suppressed)': 'CW-II (H-Suppressed)',
    'CW-mix (H-Suppressed)': 'CW-mix (H-Suppressed)',
    'CW-I (H-Included)': 'CW-I (H-Included)',
    'CW-II (H-Included)': 'CW-II (H-Included)',
    'CW-mix (H-Included)': 'CW-mix (H-Included)',
}

# --- 12 properties: (code, Excel column header, unit) ----------------------
# IMPORTANT: the 2nd element here (Excel column header) MUST match the
# ACTUAL column name in the Excel file exactly - DO NOT CHANGE this, or the
# data cannot be read. This name is also used directly as the heading text
# in the report (since the property name never appears on any figure
# anyway, only the group name does - see the ENGLISH_GROUP_NAME note above
# - there is no need for a separate English name table for it).
PROPERTIES = {
    '1':  ('bp',      'bp - Boiling Point (°C)',        '°C'),
    '2':  ('MW',       'MW - Molecular Weight',            'g/mol'),
    '3':  ('nD',       'nD - Refractive Index',              ''),
    '4':  ('epsilon',  'epsilon - Dielectric Constant',      ''),
    '5':  ('Tc',       'Tc - Critical Temperature (K)',         'K'),
    '6':  ('Vc',       'Vc - Critical Volume (cm3/mol)',      'cm3/mol'),
    '7':  ('Pc',       'Pc - Critical Pressure (MPa)',         'MPa'),
    '8':  ('logP',     'logP',                              ''),
    '9':  ('vp',       'vp - Vapor Pressure (kPa)',          'kPa'),
    '10': ('eta',      'eta - Viscosity (mPa.s)',           'mPa.s'),
    '11': ('gamma',    'gamma - Surface Tension (mN/m)',    'mN/m'),
    '12': ('lambda',   'lambda - Thermal Conductivity (W/m.K)', 'W/m.K'),
}

MOLECULE_COLUMN_PROPERTIES = 'Aliphatic Alcohol'   # molecule column in the All_Properties sheet
MOLECULE_COLUMN_GROUP = 'Solvent'               # molecule column in the CW-* sheets

# --- Fixed random seed ------------------------------------------------------
# WHY FIXED: Y-randomization AND k-fold both involve randomness. Without a
# fixed seed, running the same data twice could give a different result -
# which would break the "same data = same report, overwriting is safe"
# principle.
RANDOM_SEED = 42

# --- Y-randomization repeat count ------------------------------------------
YRAND_REPEATS = 1000

# --- k-fold thresholds -------------------------------------------------------
# WHY INDEPENDENT: these thresholds address a concern COMPLETELY SEPARATE
# from the variable-count formula (above) - "is there enough molecules in
# each fold".
K5_MIN_N = 15
K10_MIN_N = 30

# --- ABOUT the MW special case (REMOVED - rationale below) -----------------
# PREVIOUS DECISION: because of the (chemistry-faculty-verified) direct
# relationship between tr(A) and MW, an ADDITIONAL "tr(A) excluded" version
# (Version B) used to also be produced for MW (Version A = tr(A) included,
# the normal process).
# WHY REMOVED: when run on the real data, Version A and Version B gave
# IDENTICAL results in ALL groups - because VIF was already eliminating
# tr(A) on its own (not only because of MW, but because the other 4
# descriptors are ALSO heavily collinear with it). So the manual exclusion
# achieved nothing beyond repeating what VIF would do anyway - and it was
# needlessly bloating the MW report (12 sub-sections instead of 6). This is
# why MW is now run EXACTLY THE SAME WAY as the other 11 properties (a
# single version, 6 groups, with tr(A) taking its normal place in the
# candidate pool and VIF left to make its own decision).

print("Settings loaded. EXCEL_PATH =", EXCEL_PATH)

## 3. Data Loading

Loads the entire Excel file (12 properties + 6 descriptor groups) at once. The code does not automatically process the 12 properties in sequence — it first loads everything into memory, then below asks you which one you want to run.

In [ ]:
def load_data(excel_path=EXCEL_PATH):
    """Loads all sheets from the Excel file and strips whitespace from molecule names."""
    properties_df = pd.read_excel(excel_path, sheet_name='All_Properties')
    properties_df[MOLECULE_COLUMN_PROPERTIES] = properties_df[MOLECULE_COLUMN_PROPERTIES].astype(str).str.strip()

    groups = {}
    for group_name, sheet_name in GROUP_SHEETS.items():
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        df[MOLECULE_COLUMN_GROUP] = df[MOLECULE_COLUMN_GROUP].astype(str).str.strip()
        groups[group_name] = df

    print(f"Loaded: {len(properties_df)} molecules, {len(groups)} descriptor groups.")
    return properties_df, groups


def prepare_property_group_data(properties_df, groups, property_column, group_name, excluded_variable=None):
    """
    Prepares the filtered X (descriptors) and Y (target) arrays for a given
    property + group combination.

    IMPORTANT: ONLY molecules that ACTUALLY have a numeric value for this
    property are used ('-' = checked/absent, blank = not yet checked - both
    are treated as missing data here and excluded). EVERY SUBSEQUENT STEP,
    including VIF, operates only on this filtered set of molecules - this
    filtering is done first, before anything else, to avoid the mistake of
    "looking at the variance of something that doesn't exist".

    excluded_variable: a general-purpose option for specifying a variable to
    drop from the candidate pool upfront (default None = nothing excluded).
    Not currently used for any property - it used to be used for tr(A) for
    MW (see the note in the settings cell), but once it became clear that
    VIF was already eliminating tr(A) on its own, this special case was
    removed and MW was changed to run as a single version like the other 11
    properties.
    """
    property_series = pd.to_numeric(properties_df[property_column], errors='coerce')
    valid_mask = property_series.notna()

    molecules = properties_df.loc[valid_mask, MOLECULE_COLUMN_PROPERTIES].reset_index(drop=True)
    y = property_series.loc[valid_mask].reset_index(drop=True)

    group_df = groups[group_name].set_index(MOLECULE_COLUMN_GROUP)
    candidates = [v for v in CANDIDATE_VARIABLES if v != excluded_variable]

    missing_molecules = [m for m in molecules if m not in group_df.index]
    if missing_molecules:
        print(f"WARNING: molecules not found in the {group_name} sheet are being skipped: {missing_molecules}")
        molecules = molecules[~molecules.isin(missing_molecules)].reset_index(drop=True)
        y = y[properties_df.loc[valid_mask, MOLECULE_COLUMN_PROPERTIES].isin(molecules).values].reset_index(drop=True)

    x = group_df.loc[molecules.values, candidates].reset_index(drop=True)
    x = x.apply(pd.to_numeric, errors='coerce')

    complete_mask = x.notna().all(axis=1)
    if not complete_mask.all():
        print(f"WARNING: {(~complete_mask).sum()} molecules have missing/non-numeric descriptor data, skipping.")
    x = x.loc[complete_mask].reset_index(drop=True)
    y = y.loc[complete_mask].reset_index(drop=True)
    molecules = molecules.loc[complete_mask].reset_index(drop=True)

    return molecules, x, y


def calculate_std_dev(y):
    """Standard deviation (SD) of the property's RAW data - does not depend on
    the model, only on Y, so it's a SINGLE number per property, not per group."""
    return float(np.std(y.values, ddof=1))

## 4. Variable Selection — VIF and Single-Variable Power Ranking

In [ ]:
def calculate_univariate_r2(x, y):
    """
    Measures how much each candidate variable explains Y ON ITS OWN (SLR).

    WHY THIS FUNCTION IS USED EVERYWHERE: when eliminating an extra variable
    in the VIF ties, when p_max is exceeded, and in the full SLR selection
    when n is very low - the SAME rule is ALWAYS used: ranking by each
    candidate's independent/standalone power, WITHOUT EVER LOOKING at the
    MLR's own (final) result. This way we systematically avoid the risk of
    "selecting based on the outcome" (data dredging).
    """
    r2s = {}
    for variable in x.columns:
        X_single = sm.add_constant(x[[variable]])
        model = sm.OLS(y, X_single).fit()
        r2s[variable] = model.rsquared
    return r2s


def calculate_vif(x):
    """Returns the VIF value of each variable for the given candidate set."""
    X_with_const = sm.add_constant(x)
    return {
        column: variance_inflation_factor(X_with_const.values, i + 1)  # +1: skip the constant term
        for i, column in enumerate(x.columns)
    }


def eliminate_variables_by_vif(x, y, threshold=VIF_THRESHOLD):
    """
    Iteratively eliminates variables with VIF > threshold: at each step the
    variable with the highest VIF is removed, VIFs are recalculated, and
    this repeats until all variables have VIF<=threshold (or only one
    variable remains).

    TIE RULE: if more than one variable has the highest VIF (difference<0.01),
    the one that is WEAKEST ON ITS OWN (lowest SLR R^2) is removed from among
    them - this decision is made with a predetermined, fixed rule, without
    looking at the MLR result.

    Returns: (remaining_variables, vif_history) - vif_history is used to
    build the "which variable was eliminated/kept and why" table in the
    report.
    """
    remaining = list(x.columns)
    history = []  # [(variable, vif_value, decision), ...]

    while len(remaining) > 1:
        vif_values = calculate_vif(x[remaining])
        max_vif = max(vif_values.values())
        if max_vif <= threshold:
            break

        candidates = [k for k, v in vif_values.items() if abs(v - max_vif) < 0.01]
        if len(candidates) > 1:
            single_r2 = calculate_univariate_r2(x[candidates], y)
            to_remove = min(single_r2, key=single_r2.get)
        else:
            to_remove = candidates[0]

        history.append((to_remove, vif_values[to_remove], f'Eliminated (VIF>{threshold})'))
        remaining.remove(to_remove)

    final_vif = calculate_vif(x[remaining]) if len(remaining) > 1 else {remaining[0]: float('nan')}
    for k in remaining:
        history.append((k, final_vif.get(k, float('nan')), 'Kept'))

    return remaining, history


def determine_model_size(n, post_vif_variables, x, y):
    """
    Topliss-Costello rule: n >= 5*p  =>  maximum allowed number of variables
    p_max = floor(n/5). If the number of variables remaining after VIF
    exceeds this ceiling, we reduce the excess with the SAME rule (weakest
    single-variable R^2).

    We NEVER "try all combinations and pick the combination that gives the
    best R^2" - that would mean selecting based on the MLR's own result
    (outcome-based selection / data dredging), which is deliberately
    rejected.
    """
    p_max = max(1, n // 5)
    selected = list(post_vif_variables)

    while len(selected) > p_max:
        single_r2 = calculate_univariate_r2(x[selected], y)
        weakest = min(single_r2, key=single_r2.get)
        selected.remove(weakest)

    return selected, p_max

## 5. Model Building (OLS)

In [ ]:
def build_model(x, y, variables):
    """Ordinary least squares (OLS) regression - the ONE deterministic
    mathematical solution for the given variable set, not a 'choice'."""
    X = sm.add_constant(x[variables])
    return sm.OLS(y, X).fit()


def write_equation(model, target_name='Y'):
    coefficients = model.params
    terms = [f"{coefficients['const']:.3f}"]
    for variable, coefficient in coefficients.items():
        if variable == 'const':
            continue
        sign = '+' if coefficient >= 0 else '-'
        terms.append(f" {sign} {abs(coefficient):.3f}·{variable}")
    return f"{target_name} = " + ''.join(terms)

## 6. Validation — LOOCV, k-fold, CCC, Y-Randomization, Applicability Domain

In [ ]:
def apply_loocv(x, y, variables):
    """
    Leave-One-Out Cross-Validation: each molecule is left out in turn, and a
    TRULY UNSEEN prediction is produced using a model built from the rest.

    WHY: R^2_fit only shows how well the model fits its OWN training data
    (calibration); it says NOTHING about generalizability to a molecule
    outside it. Q^2LOO is computed from genuinely "unseen" predictions, so
    it is a much more reliable indicator of validity - it is ALWAYS
    applied.
    """
    n = len(y)
    predictions = np.zeros(n)
    for i in range(n):
        train_idx = [j for j in range(n) if j != i]
        X_train = sm.add_constant(x.iloc[train_idx][variables], has_constant='add')
        model = sm.OLS(y.iloc[train_idx], X_train).fit()

        X_test = sm.add_constant(x.iloc[[i]][variables], has_constant='add')
        X_test = X_test.reindex(columns=X_train.columns, fill_value=1.0)
        predictions[i] = model.predict(X_test).values[0]

    residuals = y.values - predictions
    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((y.values - y.values.mean()) ** 2))
    q2_loo = 1 - ss_res / ss_tot
    s_loo = float(np.sqrt(ss_res / n))
    return q2_loo, s_loo, predictions


def apply_kfold_cv(x, y, variables, k, seed=RANDOM_SEED):
    """
    k-fold cross-validation - added as an EXTRA check on top of LOOCV.

    WHY NEEDED AS AN EXTRA: LOOCV can sometimes be an "easy exam", especially
    if the dataset contains molecules that are very similar to each other
    (like isomers) - since k-fold leaves out more molecules at the same
    time, it provides an extra check against this kind of "cheat sheet"
    effect.

    FIXED SEED: which fold each molecule falls into is determined by a
    fixed random_seed - running it again on the same data gives the EXACT
    SAME result (overwriting the file is safe).
    """
    n = len(y)
    rng = np.random.default_rng(seed)
    indices = np.arange(n)
    rng.shuffle(indices)
    folds = np.array_split(indices, k)

    predictions = np.zeros(n)
    for fold in folds:
        test_idx = fold
        train_idx = np.setdiff1d(indices, test_idx)

        X_train = sm.add_constant(x.iloc[train_idx][variables], has_constant='add')
        model = sm.OLS(y.iloc[train_idx], X_train).fit()

        X_test = sm.add_constant(x.iloc[test_idx][variables], has_constant='add')
        X_test = X_test.reindex(columns=X_train.columns, fill_value=1.0)
        predictions[test_idx] = model.predict(X_test).values

    residuals = y.values - predictions
    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((y.values - y.values.mean()) ** 2))
    q2_k = 1 - ss_res / ss_tot
    s_k = float(np.sqrt(ss_res / n))
    return q2_k, s_k


def calculate_ccc(actual, predicted):
    """
    Concordance Correlation Coefficient (Lin, 1989). Unlike R^2, it measures
    not just correlation but whether the predictions ACTUALLY coincide 1:1
    with the actual values (on the 45-degree line) - if there is a
    systematic shift (bias), CCC comes out low, which R^2 may not catch.
    Computed here from the LOOCV predictions (not in-sample).
    """
    mean_actual, mean_predicted = float(np.mean(actual)), float(np.mean(predicted))
    var_actual, var_predicted = float(np.var(actual)), float(np.var(predicted))
    covariance = float(np.mean((actual - mean_actual) * (predicted - mean_predicted)))
    return (2 * covariance) / (var_actual + var_predicted + (mean_actual - mean_predicted) ** 2)


def apply_y_randomization(x, y, variables, repeats=YRAND_REPEATS, seed=RANDOM_SEED):
    """
    We randomly shuffle the Y values (breaking the molecule-target pairing)
    and repeatedly rebuild a model with the SAME variable set (default: 1000
    times). If the actual model's R^2 is (far) above this random
    distribution, we get STRONG, ASSUMPTION-FREE (empirical) evidence that
    the relationship is not due to chance - it answers the question the
    classic F/P-test asks, without relying on assumptions.

    FIXED SEED: so that running it again on the same data produces the
    exact same distribution (and hence the same histogram).
    """
    rng = np.random.default_rng(seed)
    actual_model = build_model(x, y, variables)
    actual_r2 = actual_model.rsquared

    random_r2s = np.zeros(repeats)
    y_array = y.values.copy()
    X = sm.add_constant(x[variables])
    for i in range(repeats):
        y_shuffled = rng.permutation(y_array)
        model = sm.OLS(y_shuffled, X).fit()
        random_r2s[i] = model.rsquared

    return actual_r2, random_r2s


def calculate_applicability_domain(x, y, variables, loocv_predictions, s_loo):
    """
    Two axes for the Williams Plot:
      - Leverage (h): computed ONLY from the descriptor (X) values, measures
        how "unusual" a molecule is in descriptor space.
        Threshold: h* = 3*p'/n  (p' = number of parameters INCLUDING the
        constant term).
      - Standardized residual: the LOOCV error divided by S_LOO. Threshold: ±3.
    """
    X = sm.add_constant(x[variables])
    X_mat = X.values
    H = X_mat @ np.linalg.pinv(X_mat.T @ X_mat) @ X_mat.T
    leverage = np.diag(H)

    p_prime = len(variables) + 1
    n = len(y)
    h_star = 3 * p_prime / n

    standardized_residual = (y.values - loocv_predictions) / s_loo if s_loo > 0 else np.zeros(n)

    outside_mask = (leverage > h_star) | (np.abs(standardized_residual) > 3)
    return leverage, standardized_residual, h_star, outside_mask

## 7. Full Analysis Orchestrator for a Single Group

Runs all the steps above in sequence and collects them into a SINGLE result dictionary.

In [ ]:
def analyze_group(molecules, x, y, sd, group_name, unit):
    """
    Runs all the steps in sequence for a given (property, group) combination:
    VIF -> model size -> OLS -> R^2_fit/S -> LOOCV -> k-fold -> CCC ->
    Y-randomization -> Applicability Domain. Collects the results into a
    single dictionary - the report functions use this dictionary.
    """
    n = len(y)

    vif_remaining, vif_history = eliminate_variables_by_vif(x, y)
    selected_variables, p_max = determine_model_size(n, vif_remaining, x, y)

    model = build_model(x, y, selected_variables)
    r2_fit = model.rsquared
    s_fit = float(np.sqrt(model.ssr / n))

    q2_loo, s_loo, loocv_predictions = apply_loocv(x, y, selected_variables)

    q2_k5 = s_k5 = None
    if n >= K5_MIN_N:
        q2_k5, s_k5 = apply_kfold_cv(x, y, selected_variables, k=5)

    q2_k10 = s_k10 = None
    if n >= K10_MIN_N:
        q2_k10, s_k10 = apply_kfold_cv(x, y, selected_variables, k=10)

    ccc = calculate_ccc(y.values, loocv_predictions)

    actual_r2, random_r2s = apply_y_randomization(x, y, selected_variables)

    leverage, standardized_residual, h_star, outside_mask = calculate_applicability_domain(
        x, y, selected_variables, loocv_predictions, s_loo
    )
    outside_molecules = [molecules.iloc[i] for i in np.where(outside_mask)[0]]

    return {
        'group_name': group_name, 'n': n, 'p_max': p_max, 'unit': unit, 'sd': sd,
        'variables': selected_variables, 'vif_history': vif_history,
        'model': model, 'equation': write_equation(model),
        'r2_fit': r2_fit, 's_fit': s_fit,
        'q2_loo': q2_loo, 's_loo': s_loo, 'loocv_predictions': loocv_predictions,
        'q2_k5': q2_k5, 's_k5': s_k5, 'q2_k10': q2_k10, 's_k10': s_k10,
        'ccc': ccc, 'actual_r2': actual_r2, 'random_r2s': random_r2s,
        'leverage': leverage, 'standardized_residual': standardized_residual, 'h_star': h_star,
        'outside_molecules': outside_molecules, 'molecules': molecules, 'y': y,
    }

## 8. Figures

In [ ]:
def plot_yrand_figure(result, title):
    # NOTE: ALL the text on this figure (title, axes, legend) is
    # deliberately written in ENGLISH because it will go into the
    # MANUSCRIPT - the rest of the code's variable/comment names are in
    # English now too, but this figure's text already had this separate,
    # strict requirement regardless.
    fig, ax = plt.subplots(figsize=(6.2, 3.6), dpi=150)
    ax.hist(result['random_r2s'], bins=30, color='#9DB8D9', edgecolor='#4472C4',
            alpha=0.85, label=f'Randomly permuted Y ({YRAND_REPEATS} trials)')
    ax.axvline(result['actual_r2'], color='#C0392B', linewidth=2.2,
               label=f"Actual model (R²={result['actual_r2']:.3f})")
    ax.set_xlabel('R² value')
    ax.set_ylabel('Number of trials (frequency)')
    ax.set_title(f'Y-Randomization Test — {title}', fontsize=10.5)
    ax.set_xlim(0, 1.0)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, linestyle='--')
    fig.tight_layout()
    return fig


def plot_williams(result, title):
    # NOTE: all the text on this figure is in English too (see the note above).
    fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=150)
    lev, res = result['leverage'], result['standardized_residual']
    outside_mask = (lev > result['h_star']) | (np.abs(res) > 3)

    ax.scatter(lev[~outside_mask], res[~outside_mask], s=26, color='#4472C4',
               alpha=0.75, edgecolor='white', linewidth=0.4, label='Training set molecules')
    ax.scatter(lev[outside_mask], res[outside_mask], s=55, color='#C0392B', marker='D',
               edgecolor='black', linewidth=0.5, label='Molecules outside AD', zorder=5)

    outside_idx = np.where(outside_mask)[0]
    for i, molecule in zip(outside_idx, result['outside_molecules']):
        ax.annotate(str(molecule), (lev[i], res[i]), textcoords='offset points',
                    xytext=(6, 4), fontsize=7.5, color='#7B241C')

    ax.axhline(3, color='gray', linestyle='--', linewidth=1)
    ax.axhline(-3, color='gray', linestyle='--', linewidth=1)
    ax.axvline(result['h_star'], color='gray', linestyle='--', linewidth=1)

    # Region labels: left/right of the h* line and inside/outside the +-3
    # lines represent four different regions (see the "Methods and Terms
    # Explanation" section in the Word report - item 10). Since the labels
    # are positioned PROPORTIONALLY to the axes (0-1 transAxes), they
    # automatically adapt to each group's own h*/data range. Written in
    # English since it's the only text on the figure (per the rule above).
    label_style = dict(fontsize=7, color='#888888', alpha=0.9, style='italic',
                        transform=ax.transAxes)
    ax.text(0.03, 0.95, 'Y-outlier', ha='left', va='top', **label_style)
    ax.text(0.03, 0.05, 'Y-outlier', ha='left', va='bottom', **label_style)
    ax.text(0.97, 0.95, 'High-risk', ha='right', va='top', **label_style)
    ax.text(0.97, 0.05, 'High-risk', ha='right', va='bottom', **label_style)
    ax.text(0.97, 0.5, 'Good leverage', ha='right', va='center', **label_style)

    ax.set_xlabel('Leverage (h)')
    ax.set_ylabel('Standardized Residual (LOOCV)')
    ax.set_title(f'Williams Plot — Applicability Domain — {title}', fontsize=10.5)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, linestyle='--')
    fig.tight_layout()
    return fig


def fig_to_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    plt.close(fig)
    return buf

## 9. Word Report — Helper Functions

This cell defines general-purpose helper functions for table/cell formatting (colors, multi-line cells, S/SD format). The report-generating functions (sections 10-11) use these.

In [ ]:
NAVY = RGBColor(0x1F, 0x38, 0x64)
ACCENT = RGBColor(0x44, 0x72, 0xC4)
GRAY = RGBColor(0x7F, 0x7F, 0x7F)
RED = RGBColor(0xC0, 0x39, 0x2B)


def _set_cell_background(cell, hex_color):
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:fill'), hex_color)
    cell._tc.get_or_add_tcPr().append(shd)


def write_cell(cell, lines, bold_first=True, header=False, gray_below=True, align=WD_ALIGN_PARAGRAPH.CENTER):
    """
    Writes multiple lines (e.g. value + S/SD note) stacked into a single
    table cell. The first line is bold/normal size, the following lines are
    smaller and gray (for extra info like S/SD).
    """
    if isinstance(lines, str):
        lines = [lines]
    cell.text = ''
    p = cell.paragraphs[0]
    p.alignment = align
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    for i, line in enumerate(lines):
        if i > 0:
            p.add_run().add_break(WD_BREAK.LINE)
        run = p.add_run(str(line))
        run.font.size = Pt(10 if i == 0 else 8)
        run.font.bold = header or (bold_first and i == 0)
        if header:
            run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        elif i > 0 and gray_below:
            run.font.color.rgb = GRAY
    if header:
        _set_cell_background(cell, '4472C4')


def create_table(doc, headers, rows, widths=None):
    """
    headers: column headers (list)
    rows: rows, each a list; each cell can be either a string or (for
          multi-line) a list.
    """
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Table Grid'
    for i, header in enumerate(headers):
        write_cell(table.rows[0].cells[i], header, header=True)
    for row in rows:
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            write_cell(row_cells[i], value)
    if widths:
        for row in table.rows:
            for i, w in enumerate(widths):
                row.cells[i].width = Inches(w)
    return table


def s_ratio_lines(value, s, unit):
    """
    Produces a two-line cell: the R^2/Q^2 value on top, and below it its own
    RMSE (S = sqrt(SS_residual/n), in the original unit) and that S's ratio
    to Y's natural standard deviation (SD) (as a %). Returns '—' if value is
    None (e.g. Q^2(k=10) was not computed because n<30). S is passed
    DIRECTLY by the caller (result['s_fit']/'s_loo'/'s_k5'/'s_k10') - it is
    NOT derived INDIRECTLY from sd (the SD with n-1) (it used to be done
    that way, which did NOT MATCH the definition of RMSE).
    """
    if value is None:
        return ['—']
    percent = round(np.sqrt(max(0.0, 1 - value)) * 100)
    return [f"{value:.3f}", f"S={s:.2f}{unit} (%{percent})"]


def add_heading(doc, text, level=1, color=NAVY):
    p = doc.add_heading(level=level)
    run = p.add_run(text)
    run.font.color.rgb = color
    return p


def add_paragraph(doc, text, italic=False, bold=False, color=None, size=11):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.font.italic = italic
    run.font.bold = bold
    if color:
        run.font.color.rgb = color
    return p

## 9b. Methods and Terms Explanation

A section written ONCE per report, explaining all the statistical methods/terms used (what - how it's calculated/works - why we chose it this way). A concrete example of the step-by-step VIF elimination process is told ONLY in the first group (see the order_no==1 check inside write_group_section below) - the same method is not repeated in the other groups, only the table is shown.

In [ ]:
def build_vif_narrative_text(vif_history, threshold):
    """Produces a text describing STEP BY STEP how the VIF elimination
    worked for one group - used ONLY in the first group of the report (as a
    concrete, worked example), not repeated in the other groups."""
    eliminated = [(d, v, k) for d, v, k in vif_history if k.startswith('Eliminated')]
    kept = [(d, v, k) for d, v, k in vif_history if k == 'Kept']
    if not eliminated:
        return (
            f"In this group, the VIF value of all the initial candidates came out below the "
            f"threshold (VIF>{threshold}), so no variable was eliminated; all candidates went "
            f"directly to the model-size (p_max) step."
        )
    sentences = ["In this group, VIF elimination worked concretely as follows:"]
    for round_no, (variable, vif_value, decision) in enumerate(eliminated, start=1):
        sentences.append(
            f"In round {round_no}, the '{variable}' variable, which had the highest VIF value "
            f"among the candidates at that point (VIF={vif_value:.1f}), was eliminated for "
            f"exceeding the threshold (VIF>{threshold})."
        )
    if kept:
        kept_names = ', '.join(f"'{d}'" for d, v, k in kept)
        sentences.append(
            f"After this round, the VIF values of the remaining variables ({kept_names}) fell "
            f"below the threshold, the process stopped here, and these variables remained to "
            f"enter the final model."
        )
    else:
        sentences.append("The process stopped after this round because only one variable remained.")
    return ' '.join(sentences)


def write_methods_and_terms_section(doc):
    """The section added to the top of every single-property report ONCE
    PER REPORT, explaining (what - how - why) all the statistical
    methods/terms used."""
    add_heading(doc, "Methods and Terms Explanation", level=1)

    add_heading(doc, "1. Why there are 6 separate groups", level=2)
    add_paragraph(doc, "What: The 5 descriptors derived from the AV-AM matrix (Per(A), A(2,1), A(2,2), "
                        "tr(A), epsilon(A)) are computed in a total of 6 different versions according to "
                        "the CW-I/CW-II/CW-mix weighting schemes and whether hydrogen atoms are included "
                        "in the matrix or not (H-Included/H-Suppressed); each group is modeled "
                        "completely independently of the others against the same target variable (Y).")
    add_paragraph(doc, "How: Separate VIF elimination, model-building, and validation steps are run for "
                        "each of the 6 separate Excel sheets (CW-I_without_H, CW-II_without_H, "
                        "CW-mix_without_H, CW-I_with_H, CW-II_with_H, CW-mix_with_H) - the "
                        "results are never mixed together at any point.")
    add_paragraph(doc, "Why this way: Different weighting/hydrogen-inclusion methods produce "
                        "conceptually different descriptor sets - combining a variable from CW-I with a "
                        "variable from CW-II in the same model would be methodologically meaningless, "
                        "because the two measure the same physical quantity under different "
                        "assumptions.")

    add_heading(doc, "2. VIF (Variance Inflation Factor)", level=2)
    add_paragraph(doc, "What: Measures how multicollinear a descriptor is with the other "
                        "descriptors.")
    add_paragraph(doc, "How: Each descriptor is made the 'target' in turn and regressed against all "
                        "the other descriptors (an auxiliary regression); VIF = 1/(1-R2_auxiliary) is "
                        "computed from this regression's R2. The variable with the highest VIF above "
                        "the threshold is removed, VIF is recalculated with the remaining variables - "
                        "this repeats iteratively until all variables fall below the threshold (or only "
                        "one variable remains). If more than one variable shares the same highest VIF "
                        "(difference<0.01), the one that is WEAKEST ON ITS OWN (lowest single-variable "
                        "R2) is removed from among them. The numeric magnitude of VIF also has a "
                        "meaning: the square root of VIF shows how many times larger a variable's "
                        "coefficient's standard error has become compared to what it would be with no "
                        "multicollinearity at all - for example, if VIF=5, the standard error has "
                        "inflated by √5≈2.2 times.")
    add_paragraph(doc, "Why this way: although many sources use VIF>10 as the threshold, because n "
                        "drops to as low as 13 for some properties in our dataset, a STRICTER "
                        "threshold (5) was chosen - being more cautious with little data increases "
                        "defensibility. The tie-break rule (weakest single-variable R2) is applied "
                        "with a predetermined, fixed rule, without ever looking at the MLR's final "
                        "result - this avoids the risk of 'selecting based on the outcome' (data "
                        "dredging). You will see below, in the first group, a concrete, step-by-step "
                        "example of how this process worked for one group; the other groups are run "
                        "with the same method.")

    add_heading(doc, "3. p_max / Topliss-Costello rule", level=2)
    add_paragraph(doc, "What: A rule that determines the maximum number of variables (p_max) that "
                        "can be used in the model.")
    add_paragraph(doc, "How: Computed with the formula p_max = floor(n/5) (n = the number of "
                        "molecules in that group). If the number of variables remaining after VIF "
                        "exceeds this ceiling, the excess is again reduced by removing the variable "
                        "with the WEAKEST single-variable R2.")
    add_paragraph(doc, "Why this way: The n>=5p constraint is a rule widely used in the literature to "
                        "prevent the model from overfitting relative to the sample size. The approach "
                        "of 'try all combinations, pick the combination that gives the best R2' is "
                        "deliberately not used, because that would mean selecting based on the MLR's "
                        "own result (data dredging).")

    add_heading(doc, "4. Model equation (OLS)", level=2)
    add_paragraph(doc, "What: The regression equation built from the selected variables using the "
                        "ordinary least squares (OLS) method.")
    add_paragraph(doc, "How: sm.OLS finds the ONE deterministic mathematical solution for the "
                        "selected variable set plus the constant term.")
    add_paragraph(doc, "Why this way: OLS is the standard, interpretable, and deterministic method "
                        "for linear relationships - the coefficients show each descriptor's "
                        "independent effect on the target when the others are held constant; this is "
                        "not a 'choice', it is the only possible linear fit for the given variables.")

    add_heading(doc, "5. R2_fit and S", level=2)
    add_paragraph(doc, "What: R2_fit shows how well the model fits its OWN training data (the same "
                        "molecules used when it was built); S is the standard error of that same fit, "
                        "in the original unit.")
    add_paragraph(doc, "How: S=√(SSR/n) (the root-mean of the sum of squared residuals). The %SD "
                        "ratio shows, as a percentage, how much of the data's natural variability the "
                        "model still cannot explain, by dividing S by the data's natural standard "
                        "deviation.")
    add_paragraph(doc, "Why this way: R2_fit is NOT sufficient ON ITS OWN, because it only shows the "
                        "fit to the training data - it says nothing about generalizability to a "
                        "molecule outside it; this is why the cross-validation steps below are also "
                        "necessary. A general guideline for the %SD ratio: the smaller the value "
                        "(e.g. 20-30%), the more the model explains of the data's natural "
                        "variability; as it approaches 70-100%, the model's explanatory power "
                        "weakens - a value close to 100% means, in practice, that the model performs "
                        "no better at predicting the data than simply stating the mean for every "
                        "molecule.")

    add_heading(doc, "6. Q2LOO", level=2)
    add_paragraph(doc, "What: The coefficient of determination computed by leaving each molecule "
                        "out in turn and producing a truly unseen prediction for it using a model "
                        "built from the remaining molecules.")
    add_paragraph(doc, "How: For each of the n molecules: a model is built with the other n-1 "
                        "molecules (excluding that one), and a prediction is produced for the "
                        "left-out molecule. Q2LOO and S_LOO are computed by comparing these n "
                        "predictions with the actual values.")
    add_paragraph(doc, "Why this way: because every molecule is evaluated with a genuinely "
                        "'unseen' prediction, it is a much more reliable indicator of external "
                        "validity than R2_fit - this is why it is ALWAYS applied in EVERY group, "
                        "regardless of the amount of data.")

    add_heading(doc, "7. Q2(k=5)/Q2(k=10)", level=2)
    add_paragraph(doc, "What: k-fold cross-validation - the molecules are split into k equal parts "
                        "(folds), and the model's predictive power is tested by leaving out one part "
                        "at a time (more than one molecule at once).")
    add_paragraph(doc, "How: for k=5 the molecules are randomly split (with a fixed seed=42) into 5 "
                        "groups, for k=10 into 10 groups; each group is used as the test set in "
                        "turn.")
    add_paragraph(doc, "Why this way: LOOCV can sometimes be an 'easy exam', especially if the "
                        "dataset contains molecules that are very similar to each other (like "
                        "isomers) - since k-fold leaves out more molecules at the same time, it "
                        "provides an EXTRA check against this kind of 'cheat sheet' effect. The "
                        "thresholds (k=5 for n>=15, k=10 for n>=30) were chosen to guarantee that "
                        "enough molecules remain in each fold.")

    add_heading(doc, "8. CCC (Concordance Correlation Coefficient)", level=2)
    add_paragraph(doc, "What: CCC (Lin, 1989) measures not only whether two variables (actual and "
                        "predicted values) move together, but also whether they ACTUALLY coincide "
                        "1:1 (on the 45-degree line) with each other.")
    add_paragraph(doc, "How: computed with the formula CCC = 2*Cov(actual, predicted) / "
                        "[Var(actual) + Var(predicted) + (mean_actual - mean_predicted)^2]; the last "
                        "term in the denominator penalizes any systematic shift (bias) of the "
                        "predictions from the actual values. Here CCC is computed from the LOOCV "
                        "predictions, not from the model's in-sample fit to its own training data.")
    add_paragraph(doc, "Why this way: Pearson correlation/R2 only measures the strength of the "
                        "linear relationship - even if the model systematically over- or "
                        "under-predicts (as long as the ratio stays the same) the correlation can "
                        "still come out high. Because CCC penalizes both precision (correlation "
                        "strength) and accuracy (closeness to the actual y=x line) together, it can "
                        "reveal a systematic shift that R2/Q2 would not catch. As a mathematical "
                        "property: CCC is always less than or equal to |Pearson correlation|; if "
                        "there is no systematic shift, CCC equals the correlation - how much lower "
                        "CCC comes out than the correlation directly reflects the magnitude of the "
                        "systematic shift in the model. This is why it is used in the methodology as "
                        "an additional, complementary layer of assurance on top of R2/Q2.")

    add_heading(doc, "9. Y-Randomization", level=2)
    add_paragraph(doc, "What: A test that randomly shuffles the Y (target) values, deliberately "
                        "breaking the molecule-target pairing, and repeatedly rebuilds the model "
                        "with the same variable set.")
    add_paragraph(doc, "How: 1000 different random shuffling trials are run (fixed seed=42), R2 is "
                        "computed for each trial to build a distribution; the actual model's R2 is "
                        "compared against this distribution.")
    add_paragraph(doc, "Why this way: it directly and empirically answers the question the classic "
                        "F/P-test asks - 'did this relationship arise by chance?' - without relying "
                        "on distributional assumptions (normality, etc.); 1000 repeats is a large "
                        "enough sample for the distribution to be estimated reliably. A different "
                        "interpretation is automatically written depending on each group's own "
                        "result (the three-tier sentence you will see below: strongly passed / "
                        "weak-interpret with caution / invalid) - this part varies group by group "
                        "and is not repeated here.")

    add_heading(doc, "10. Applicability Domain / Williams Plot", level=2)
    add_paragraph(doc, "General purpose: A regression model can only produce a reliable prediction "
                        "for molecules 'similar' to the ones it saw during training. The Williams "
                        "Plot visually answers the question 'how much can we trust the model's "
                        "prediction for this molecule?' by examining each molecule from two "
                        "different angles: (1) how unusual this molecule's descriptor values are "
                        "compared to the other molecules, (2) how large the model's prediction "
                        "error is for this molecule.")
    add_paragraph(doc, "1. Leverage (h) - 'how unusual are this molecule's descriptors?'", bold=True)
    add_paragraph(doc, "What: Leverage measures how 'extreme/unusual' a molecule's position is "
                        "relative to the other molecules by looking ONLY at its descriptor (X) "
                        "values - without ever using the target value (Y). Conceptually: if a "
                        "molecule's values for descriptors like Per(A), A(2,1) sit right in the "
                        "middle of the 'typical cloud' formed by all the other molecules, leverage "
                        "is low; if it sits at the edge of this cloud, or even outside it, leverage "
                        "is high.")
    add_paragraph(doc, "How: the H = X(X'X)^-1 X' matrix is computed (called the 'hat matrix' "
                        "because it puts a 'hat' on the predictions); each molecule's leverage is "
                        "the corresponding diagonal element of this matrix (h_i). This is not simply "
                        "a 'distance from the others' - it is a kind of weighted distance computed "
                        "according to how the descriptors relate to each other (taking the "
                        "correlation structure into account). Threshold: h*=3p'/n (p' = the number "
                        "of parameters including the constant term, n = the number of molecules). "
                        "It has a mathematical property: the average leverage over ALL molecules is "
                        "always exactly equal to p'/n - in other words, h* represents '3 times the "
                        "average'.")
    add_paragraph(doc, "Why this way: the 3x average-leverage rule is a standard threshold widely "
                        "used in Applicability Domain analyses in the QSAR/QSPR literature.")
    add_paragraph(doc, "2. Standardized Residual - 'how wrong is the model's prediction for this "
                        "molecule?'", bold=True)
    add_paragraph(doc, "What: the difference between the LOOCV prediction (the model's prediction "
                        "made WITHOUT ever seeing this molecule) and the actual value for this "
                        "molecule, scaled relative to the 'typical' error size.")
    add_paragraph(doc, "How: (actual value - LOOCV prediction) / S_LOO. Since S_LOO is the typical "
                        "size of the LOOCV errors across all molecules, this division standardizes "
                        "the raw error as 'how many typical-error-units it has deviated by' (like a "
                        "z-score). Threshold ±3: if the errors are roughly normally distributed, "
                        "99.7% of the values should fall within ±3 standard deviations.")
    add_paragraph(doc, "Why this way: the ±3 threshold is the classic 3-sigma rule widely used in "
                        "statistics; computing it from LOOCV means that here too we are looking at "
                        "the error size of a genuinely 'unseen' prediction.")
    add_paragraph(doc, "3. Reading the two together: four regions", bold=True)
    add_paragraph(doc, "Since the Williams Plot has leverage on the X axis and standardized "
                        "residual on the Y axis, every molecule falls into one of four categories:")
    add_paragraph(doc, "- Low leverage + small residual -> the 'normal', reliable region for the "
                        "model.")
    add_paragraph(doc, "- High leverage + small residual -> its descriptors are unusual, but the "
                        "model still predicts it correctly (a 'benign' extreme point).")
    add_paragraph(doc, "- Low leverage + large residual -> its descriptors are not unusual at all, "
                        "but the model predicts this molecule incorrectly - generally the most "
                        "noteworthy case (could be a data error or a chemical property the model is "
                        "not capturing).")
    add_paragraph(doc, "- High leverage + large residual -> both its descriptors are extreme and "
                        "its prediction error is large - the riskiest group.")
    add_paragraph(doc, "This distinction matters because, instead of just saying 'inside/outside "
                        "the AD', it lets us distinguish WHY a molecule falls outside (in "
                        "descriptor space, in prediction accuracy, or in both).")
    add_paragraph(doc, "Finding the four regions on the plot: the dashed lines on the plot mark "
                        "exactly these four regions: the vertical dashed line marks the h* "
                        "threshold, and the two horizontal dashed lines mark the +3 and -3 "
                        "boundaries. Points to the left of the vertical line and between the "
                        "horizontal lines are in the normal region; to the right but between the "
                        "horizontal lines are benign extreme points; to the left but outside the "
                        "horizontal lines are genuine outliers; to the right AND outside the "
                        "horizontal lines are the riskiest group. The points marked with a red "
                        "diamond (outside the Applicability Domain) are molecules falling into any "
                        "of these three categories other than the normal region. On the figure "
                        "itself these regions are also marked with small English labels ('Good "
                        "leverage', 'Y-outlier', 'High-risk').", italic=True)

    doc.add_page_break()

## 10. Single Property Report (Options 1-12)

Writes a detailed Word section for a group (VIF table, equation, performance table, Y-randomization and Williams Plot figures) and also prints that group's results to the screen.

In [ ]:
def write_group_section(doc, result, order_no, also_print=True):
    """Adds a group's FULL result to the Word document (VIF, equation,
    table, Y-randomization, Williams Plot) - and optionally prints it to
    the screen too."""
    title = f"{order_no}. {result['group_name']}"
    add_heading(doc, title, level=1)

    if also_print:
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        print(f"n={result['n']}  p_max={result['p_max']}  selected variables={result['variables']}")
        print(f"Equation: {result['equation']}")
        print(f"R2_fit={result['r2_fit']:.3f}  Q2LOO={result['q2_loo']:.3f}  CCC={result['ccc']:.3f}")

    add_paragraph(doc, f"VIF (Variance Inflation Factor) Analysis — threshold: VIF > {VIF_THRESHOLD}", bold=True)
    vif_rows = [[d, ('—' if (isinstance(v, float) and np.isnan(v)) else f"{v:.1f}"), k]
                 for d, v, k in result['vif_history']]
    create_table(doc, ['Variable', 'VIF Value', 'Decision'], vif_rows)

    if order_no == 1:
        # We describe HOW the VIF elimination worked step by step ONLY for
        # the first group, as a concrete example (see the Methods and Terms
        # Explanation, item 2) - the same narrative is NOT REPEATED in the
        # later groups, only the table is shown.
        add_paragraph(doc, build_vif_narrative_text(result['vif_history'], VIF_THRESHOLD), italic=True, size=10)

    doc.add_paragraph()
    add_paragraph(doc, f"Selected variables (after VIF/p_max): {', '.join(result['variables'])}", bold=True)
    add_paragraph(doc, f"Model equation: {result['equation']}")

    add_paragraph(doc, "Model Performance  (S = RMSE, in the original unit; %value = S's ratio to SD)", bold=True)
    unit = result['unit']
    performance_rows = [[
        str(result['n']),
        s_ratio_lines(result['r2_fit'], result['s_fit'], unit),
        s_ratio_lines(result['q2_loo'], result['s_loo'], unit),
        s_ratio_lines(result['q2_k5'], result['s_k5'], unit),
        s_ratio_lines(result['q2_k10'], result['s_k10'], unit),
        f"{result['ccc']:.3f}",
    ]]
    create_table(doc, ['n', 'R2_fit', 'Q2LOO', 'Q2(k=5)', 'Q2(k=10)', 'CCC'], performance_rows)
    doc.add_paragraph()

    add_paragraph(doc, "Y-Randomization Test", bold=True)
    mean_random_r2 = float(np.mean(result['random_r2s']))
    max_random_r2 = float(np.max(result['random_r2s']))
    random_array = np.asarray(result['random_r2s'])
    # Empirical "p-value": what fraction of the random trials managed to
    # produce an R^2 EQUAL TO OR HIGHER than the actual model's R^2. The
    # LOWER this fraction, the more clearly the model stands apart from
    # random noise. IMPORTANT: the text below is now CONDITIONAL - it
    # writes DIFFERENT sentences depending on the actual comparison result.
    # (The old version always said "shows the model is not due to chance";
    # this was a WRONG and misleading statement for weak groups where R^2
    # stayed BELOW EVEN THE AVERAGE of the random distribution - seen in
    # some groups for low-n properties. Now the text is ALWAYS consistent
    # with what is seen in the figure.)
    exceed_fraction = float(np.mean(random_array >= result['actual_r2']))
    if exceed_fraction <= 0.05:
        yrand_text = (
            f"The actual model's R2 value ({result['actual_r2']:.3f}) was matched or exceeded "
            f"by only %{exceed_fraction * 100:.1f} of the {YRAND_REPEATS} random Y-shuffling "
            f"trials (random distribution mean={mean_random_r2:.3f}, max={max_random_r2:.3f}) — "
            f"supporting that the model's correlation is NOT due to CHANCE."
        )
    elif result['actual_r2'] > mean_random_r2:
        yrand_text = (
            f"The actual model's R2 value ({result['actual_r2']:.3f}) is above the mean of the "
            f"random Y-shuffling trials (mean={mean_random_r2:.3f}), but %{exceed_fraction * 100:.1f} "
            f"of the trials still managed to produce an equal or higher value — this shows that "
            f"the separation between the model and chance is NOT STRONG, and this group's results "
            f"should be interpreted with caution."
        )
    else:
        yrand_text = (
            f"The actual model's R2 value ({result['actual_r2']:.3f}) was EQUAL TO OR BELOW the "
            f"mean of the random Y-shuffling trials (mean={mean_random_r2:.3f}) — meaning the "
            f"model built for this group is statistically INDISTINGUISHABLE from a chance "
            f"correlation and should NOT be considered a VALID/RELIABLE model."
        )
    add_paragraph(doc, yrand_text)
    yrand_title_en = ENGLISH_GROUP_NAME.get(result['group_name'], result['group_name'])
    yrand_fig = plot_yrand_figure(result, yrand_title_en)
    doc.add_picture(fig_to_bytes(yrand_fig), width=Inches(5.4))

    add_paragraph(doc, "Applicability Domain (Williams Plot)", bold=True)
    outside_list = result['outside_molecules']
    if outside_list:
        add_paragraph(doc, f"A total of {len(outside_list)} molecules fall outside the leverage "
                            f"(h*={result['h_star']:.3f}) or standardized residual (±3σ) limits: "
                            f"{', '.join(map(str, outside_list))}.")
    else:
        add_paragraph(doc, "No molecules in this group fall outside the Applicability Domain.")
    williams_fig = plot_williams(result, yrand_title_en)
    doc.add_picture(fig_to_bytes(williams_fig), width=Inches(5.4))

## 11. Full Report Generation for a Single Property (Main Function — Options 1-12)

In [ ]:
def _build_group_list(property_code):
    """
    Returns the work list that runs EACH of the 6 descriptor groups as a
    single version, for a given property.

    NOTE (past decision - changed): previously, for MW, because of tr(A)'s
    direct (chemistry-faculty-verified) relationship with MW, every group
    used to be run as TWO VERSIONS (tr(A) included / tr(A) excluded). When
    tested on the real data, the two versions gave IDENTICAL results IN
    EVERY GROUP - because VIF was already eliminating tr(A) on its own (not
    only because of MW, but because the other 4 descriptors are ALSO
    heavily collinear with it). Since the manual exclusion achieved nothing
    beyond repeating what VIF would do anyway, this special case was
    removed - MW now runs EXACTLY THE SAME WAY as the other 11 properties
    (a single version, with VIF left to make its own decision).
    """
    return [(group_name, group_name, None) for group_name in GROUP_SHEETS]


def run_single_property(properties_df, groups, property_code, property_column, unit):
    """
    Analyzes all the groups (12 for MW, 6 for the others) for the single
    selected property, both prints to the screen and saves them into a
    single Word file.
    """
    print(f"\n{property_column} analysis starting...\n")

    doc = Document()
    add_heading(doc, property_column.upper() + " — REGRESSION ANALYSIS", level=0)
    add_paragraph(doc, "Results Report", italic=True)

    work_list = _build_group_list(property_code)
    results = []

    for label, group_name, excluded in work_list:
        molecules, x, y = prepare_property_group_data(
            properties_df, groups, property_column, group_name, excluded_variable=excluded
        )
        if len(y) < 5:
            print(f"WARNING: not enough data for {label} (n={len(y)}), skipping.")
            continue
        sd = calculate_std_dev(y)
        result = analyze_group(molecules, x, y, sd, label, unit)
        results.append(result)

    # --- Cover info table ---
    first_n = results[0]['n'] if results else '—'
    first_sd = results[0]['sd'] if results else '—'
    cover_rows = [
        ['Property Examined', property_column],
        ['Total Number of Molecules (n)', str(first_n)],
        ['Natural Standard Deviation of the Data (SD)', f"±{first_sd:.2f}{unit}" if isinstance(first_sd, float) else str(first_sd)],
        ['Validation Steps', f'LOOCV (always) . k=5 (n>={K5_MIN_N}) . k=10 (n>={K10_MIN_N}) . Y-Randomization . CCC . Williams Plot (AD)'],
        ['Number of Groups Analyzed', str(len(work_list))],
    ]
    create_table(doc, ['Info', 'Value'], cover_rows)
    doc.add_page_break()

    # --- Methods and Terms Explanation (ONCE per report) ---
    write_methods_and_terms_section(doc)

    # --- Overall comparison (among this property's own groups) ---
    add_heading(doc, "Overall Comparison of the Groups", level=1)
    summary_rows = [[
        s['group_name'], ', '.join(s['variables']),
        s_ratio_lines(s['r2_fit'], s['s_fit'], s['unit']),
        s_ratio_lines(s['q2_loo'], s['s_loo'], s['unit']),
        f"{s['ccc']:.3f}",
    ] for s in results]
    create_table(doc, ['Group', 'Variables', 'R2_fit', 'Q2LOO', 'CCC'], summary_rows)
    doc.add_page_break()

    # --- Detailed sections per group ---
    add_heading(doc, "Detailed Results by Group", level=1)
    for i, result in enumerate(results, start=1):
        write_group_section(doc, result, i)

    # --- Overall assessment ---
    doc.add_page_break()
    add_heading(doc, "Overall Assessment", level=1)
    if results:
        best = max(results, key=lambda s: s['q2_loo'])
        add_paragraph(doc, f"The highest Q2LOO value was observed in the {best['group_name']} group (Q2LOO={best['q2_loo']:.3f}).")

    file_name = f"{property_code}_Analysis.docx"
    doc.save(file_name)
    print(f"\nReport saved: {file_name}")
    return results

## 13. Main Menu — Run Entry Point

When you run the cell below, you will be asked which property (1-12) you want. If you want to run it again for a different property, simply run this cell again.

In [ ]:
def show_menu():
    print("Which property would you like to analyze?\n")
    for key, (code, label, unit) in PROPERTIES.items():
        print(f"  {key}) {label}")


def run():
    properties_df, groups = load_data()
    show_menu()
    choice = input("\nYour choice (1-12): ").strip()

    if choice in PROPERTIES:
        code, column, unit = PROPERTIES[choice]
        return run_single_property(properties_df, groups, code, column, unit)
    else:
        print("Invalid choice. Please enter a number from 1 to 12.")
        return None


# Run the line below to execute:
result = run()